In [1]:
import os
import json
import ast
import re
from typing import Dict, List

import numpy as np
import pandas as pd
import evaluate
from datasets import load_dataset
from tqdm import tqdm

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

/home/rayane/Desktop/Paris Cité/M1/PPD/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "JungZoona/T3Q-qwen2.5-14b-v1.0-e3"

DATA_TRAIN = "data/train.csv"
DATA_VALID = "data/validation.csv"

SEED = 42


MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 32
BATCH_SIZE = 1  
DEVICE = "cuda"

In [3]:
def parse_answers_field(x):
    s = str(x).strip()
    s2 = re.sub(r"array\(\s*(\[[\s\S]*?\])\s*,\s*dtype=[^)]+\)", r"\1", s)
    ans = ast.literal_eval(s2)
    ans["text"] = list(ans["text"])
    ans["answer_start"] = [int(v) for v in ans["answer_start"]]
    return ans

def normalize_dataset(example):
    example["answers"] = parse_answers_field(example["answers"])
    example["id"] = str(example["id"])
    return example

In [4]:
def build_prompt(question: str, tokenizer) -> str:
    question = str(question).strip()
    system_msg = (
        "You are a helpful assistant. "
        "Answer the question with a short phrase. "
        "Do not add explanations."
    )
    user_msg = f"Question: {question}\nAnswer:"

    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    return f"{system_msg}\n\n{user_msg}"

def clean_generation(text: str) -> str:
    t = (text or "").strip()
    # keep first line only (often enough for short answers)
    t = t.split("\n")[0].strip()
    # remove common prefixes
    for pref in ["Answer:", "A:", "assistant:", "Assistant:"]:
        if t.lower().startswith(pref.lower()):
            t = t[len(pref):].strip()
    return t

In [ ]:
set_seed(SEED)

ds = load_dataset("csv", data_files={"train": DATA_TRAIN, "validation": DATA_VALID})
ds = ds.map(normalize_dataset)

# 4-bit quantization (to fit 14B on 8GB)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.eval()

eval_split = ds["validation"]

squad_metric = evaluate.load("squad")
predictions = []

ids = eval_split["id"]
questions = eval_split["question"]

for start in tqdm(range(0, len(eval_split), BATCH_SIZE), desc="Generating (No-RAG, Qwen)"):
    batch_ids = ids[start:start + BATCH_SIZE]
    batch_q = questions[start:start + BATCH_SIZE]

    prompts = [build_prompt(q, tokenizer) for q in batch_q]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )

    # Move inputs to GPU if single-device; device_map="auto" on one GPU is OK
    if DEVICE == "cuda":
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
            num_beams=1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    gen_ids = out[:, prompt_len:]
    gen_texts = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

    for ex_id, gt in zip(batch_ids, gen_texts):
        predictions.append({"id": str(ex_id), "prediction_text": clean_generation(gt)})

references = [{"id": str(ex["id"]), "answers": ex["answers"]} for ex in eval_split]
results = squad_metric.compute(predictions=predictions, references=references)

df = pd.DataFrame([{
    "split": "validation",
    "exact_match": results["exact_match"],
    "f1": results["f1"],
    "model": MODEL_NAME,
}])

print("\n=== No-RAG (Qwen) Validation Results ===")
print(df.to_string(index=False))

os.makedirs("results", exist_ok=True)
df.to_csv("results/metrics_table_qwen_no_rag.csv", index=False)
with open("results/metrics_qwen_no_rag.json", "w") as f:
    json.dump(results, f, indent=2)

# Save a small sample of predictions for inspection
sample_path = "results/sample_predictions_qwen_no_rag.jsonl"
with open(sample_path, "w") as f:
    for row in predictions[:200]:
        f.write(json.dumps(row) + "\n")

print("\nSaved:")
print(" - results/metrics_table_qwen_no_rag.csv")
print(" - results/metrics_qwen_no_rag.json")
print(" - results/sample_predictions_qwen_no_rag.jsonl")
